In [1]:
!pip install asteroid
import torch
import os
from torch.utils.data import DataLoader
from torch.optim import Adam
from asteroid.models import ConvTasNet
from asteroid.data.librimix_dataset import LibriMix
from asteroid.losses import PITLossWrapper
from asteroid.losses.sdr import PairwiseNegSDR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
model = ConvTasNet(
    n_src=3,
    sample_rate=16000,
    kernel_size=32,
    n_filters=512,
    stride=16,
    bn_chan=128,
    hid_chan=512,
    mask_act='relu',
    n_blocks=8,
    n_repeats=3,
    skip_chan=128
)
if torch.cuda.device_count() > 1:
    print(f"Đang sử dụng {torch.cuda.device_count()} GPU")
    model = torch.nn.DataParallel(model)

model = model.to(device)
def unwrap(m):
    return m.module if isinstance(m, torch.nn.DataParallel) else m

def load_state(model, state_dict):
    if any(k.startswith('module.') for k in state_dict):
        state_dict = {k.replace('module.', '', 1): v for k, v in state_dict.items()}
    unwrap(model).load_state_dict(state_dict)

Đang sử dụng 2 GPU


In [3]:

train_set = LibriMix(
    csv_dir="/kaggle/input/datasets/habangchu/mixture-train-mix-clean-csv",
    task="sep_clean",
    sample_rate=16000,
    n_src=3,
    segment=3
)

val_set = LibriMix(
    csv_dir="/kaggle/input/datasets/habangchu/dev-path",
    task="sep_clean",
    sample_rate=16000,
    n_src=3,
    segment=3
)

train_loader = DataLoader(train_set, batch_size=4, shuffle=True,num_workers=4,pin_memory=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=4, shuffle=False,num_workers=4,pin_memory=True, drop_last=True)


Drop 0 utterances from 33900 (shorter than 3 seconds)
Drop 0 utterances from 3000 (shorter than 3 seconds)


In [4]:
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=0.0)
loss_func = PITLossWrapper(PairwiseNegSDR(sdr_type="sisdr"), pit_from='pw_mtx')

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

best_val_loss = float('inf')
patience_counter = 0
early_stop_patience = 10 


In [5]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
import os
import time
import numpy as np

input_checkpoint = "/kaggle/input/models/habangchu/hdhd/pytorch/default/1/last_checkpoint.pth" 
output_checkpoint = "/kaggle/working/last_checkpoint.pth"

epochs = 200
history = {'train_loss': [], 'val_loss': []}
start_epoch = 0
best_val_loss = float('inf')
patience_counter = 0
early_stop_patience = 10

path_to_load = None
if os.path.exists(output_checkpoint):
    path_to_load = output_checkpoint
    print("Load checkpoint từ Working...")
elif os.path.exists(input_checkpoint):
    path_to_load = input_checkpoint
    print("Load checkpoint từ Input...")

if path_to_load:
    checkpoint = torch.load(path_to_load, map_location=device)
    load_state(model, checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_val_loss = checkpoint['best_val_loss']
    print(f"Tiếp tục từ Epoch {start_epoch}")


for epoch in range(start_epoch, epochs):
    start_time = time.time()
    
    model.train()
    train_loss = 0
    for i, (mixture, sources) in enumerate(train_loader):
        mixture, sources = mixture.to(device), sources.to(device)
        optimizer.zero_grad()
        est_sources = model(mixture)
        loss = loss_func(est_sources, sources)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        train_loss += loss.item()
        
        if i % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{i}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    avg_train_loss = train_loss / len(train_loader)
    

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for mixture, sources in val_loader:
            mixture, sources = mixture.to(device), sources.to(device)
            est_sources = model(mixture)
            v_loss = loss_func(est_sources, sources)
            val_loss += v_loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f" Kết thúc Epoch {epoch+1} - Avg Train Loss: {avg_train_loss:.4f} | Avg Val Loss: {avg_val_loss:.4f}")

    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    plt.figure(figsize=(10, 6))
    plt.plot(history['train_loss'], label='Train Loss', color='blue', linewidth=1.5)
    plt.plot(history['val_loss'], label='Val Loss', color='red', linewidth=1.5)
    plt.title(f'Conv-TasNet SI-SNR Loss (Epoch {epoch+1})')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (dB)')
    plt.legend()
    plt.grid(True, linestyle='--')
    plt.savefig('/kaggle/working/loss_plot.png', dpi=300) 
    plt.close()

    scheduler.step(avg_val_loss)

  

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(unwrap(model).state_dict(), "/kaggle/working/best_model.pth")
        patience_counter = 0
        print(f"Đã lưu model tốt nhất tại Epoch {epoch+1}")
    else:
        patience_counter += 1
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': unwrap(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'best_val_loss': best_val_loss
    }
    torch.save(checkpoint, output_checkpoint)
    epoch_time = (time.time() - start_time) / 60
    print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {epoch_time:.1f} min")

    if patience_counter >= early_stop_patience:
        print("Early stopping triggered!")
        break

pd.DataFrame(history).to_csv("/kaggle/working/training_history.csv", index=False)


Load checkpoint từ Input (dữ liệu bạn upload)...
Tiếp tục thành công từ Epoch 200
